# 13.5 · 条件 GAN / Conditional GAN (cGAN)

> **课程定位 / Where this fits**
> 第 5 课，**Part 13 · 生成模型**。从"随机生成"到"**可控生成**"。
> Lesson 5, **Part 13 · Generative Models**. From "random generation" to "**controllable generation**."
>
> 前面的 GAN 只能"随便生成一个数字", 没法指定"我就要一个 7"。**条件 GAN(cGAN)** 加一个简单却强大的改造：**把条件(如数字标签)同时喂给生成器和判别器**, 于是可以**按指令生成**——你说要哪个数字, 它就生成哪个。这是**可控生成**的基石, 也是 **pix2pix(边缘图→照片)、文生图(文本→图像)** 等一切"按条件生成"任务的核心思想。本课从零搭 cGAN, 在 MNIST 上训练, **指定生成各个数字**, 并用一个分类器**定量验证可控性**。
> Earlier GANs only "generate a random digit," with no way to request "a 7." A **conditional GAN (cGAN)** adds a simple yet powerful change: **feed the condition (e.g. a digit label) to both generator and discriminator**, enabling **generation on demand** — ask for a digit and it produces that digit. This is the cornerstone of **controllable generation** and the core idea behind **pix2pix (edges→photo), text-to-image**, and all "generate given a condition" tasks. We build a cGAN from scratch, train on MNIST, **generate specified digits**, and **quantify controllability** with a classifier.
>
> 💼 **实战/面试视角**："cGAN 怎么加条件 / 判别器为什么也要条件 / 可控生成应用(pix2pix/文生图) / 怎么评估可控性" 是可控生成常考。
> 💼 **Practical/interview angle:** "how cGAN adds the condition / why the discriminator is also conditioned / controllable apps (pix2pix/text-to-image) / evaluating controllability" — common.

> 📐 **符号约定 / Notation**
> - 条件 $y$ —— 想要生成的类别/属性(这里是数字标签) / the condition (digit label here)
> - $G(z, y)$ / $D(x, y)$ —— 生成器/判别器都接收条件 / both take the condition

> 💡 **面试相关 / Interview-relevant**
> - "cGAN 相比普通 GAN 的改造"（出镜率 ★★★★）
> - "为什么判别器也要看条件"（★★★★★）
> - "可控生成的应用(pix2pix/cycleGAN/文生图)"（★★★★）
> - "怎么评估生成是否符合条件"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解如何给 GAN 加条件实现可控生成。
   Understand how to condition a GAN for controllable generation.
2. 理解**判别器为何也要接收条件**(否则生成器可作弊)。
   Understand why the discriminator is also conditioned.
3. **从零搭 cGAN** 训练, 指定生成各个数字。
   Build a cGAN from scratch and generate specified digits.
4. 用分类器**定量评估可控性**, 并诚实看待局限。
   Quantify controllability with a classifier and honestly assess limits.

## 目录 / TOC
1. [从随机到可控 ⭐](#1)
2. [怎么加条件 ⭐](#2)
3. [从零搭 cGAN：指定生成数字 ⭐](#3)
4. [评估可控性 + 应用 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 从随机到可控 ⭐ / From Random to Controllable

普通 GAN 的生成器 $G(z)$ 只吃一个**随机噪声 $z$**, 输出一张随机的图——你无法控制它生成哪个数字(噪声不同就是不同数字, 但你不知道哪个噪声对应哪个数字)。
A plain GAN's generator $G(z)$ takes only **random noise $z$** and outputs a random image — you can't control which digit (different noise gives different digits, but you don't know which noise maps to which).

**条件 GAN** 让生成器多吃一个**条件 $y$**(比如"我要数字 7"这个标签): $G(z, y)$。现在 $z$ 控制"长什么样的 7"(粗细、风格), $y$ 控制"是哪个数字"。这样就能**指定生成**: 想要 7 就喂 $y=7$。
A **conditional GAN** gives the generator an extra **condition $y$** (e.g. the label "I want a 7"): $G(z, y)$. Now $z$ controls "what kind of 7" (thickness, style), and $y$ controls "which digit." So you can **generate on demand**: feed $y=7$ to get a 7.

条件可以是任何东西：类别标签(本课)、一段文字(→文生图)、另一张图(→pix2pix 边缘→照片、CycleGAN 风格迁移)、属性向量等。这把生成模型从"玩具"变成了**可用的工具**。
The condition can be anything: a class label (here), text (→ text-to-image), another image (→ pix2pix edges→photo, CycleGAN style transfer), an attribute vector, etc. This turns generative models from "toys" into **usable tools**.


<a id="2"></a>
## 2. 怎么加条件 ⭐ / How to Add the Condition

最简单的做法：**把条件 $y$(转成向量, 如 one-hot)拼接到输入上**。
The simplest approach: **concatenate the condition $y$ (as a vector, e.g. one-hot) to the input.**
- **生成器**：输入 = 噪声 $z$ ⊕ 标签 $y$ → 生成对应类别的图。
  **Generator:** input = noise $z$ ⊕ label $y$ → image of that class.
- **判别器**：输入 = 图 $x$ ⊕ 标签 $y$ → 判断"这张图**既要像真的、又要确实是标签 $y$ 那个类别**"。
  **Discriminator:** input = image $x$ ⊕ label $y$ → judge "is this both **realistic AND actually class $y$**."

**关键(面试核心): 为什么判别器也必须看条件?** 如果判别器只看图、不看标签, 那生成器可以**作弊**——不管你要哪个数字, 它都生成一个最容易骗过判别器的数字(比如总生成"1"), 判别器觉得"是真数字"就放行, 但它**完全没遵守条件**。只有让判别器**同时检查"图是否匹配标签"**, 才能逼生成器**老老实实按条件生成**。
**Key (interview): why must the discriminator also see the condition?** If the discriminator only sees the image, the generator can **cheat** — regardless of the requested digit, generate whatever best fools the discriminator (e.g. always a "1"); the discriminator says "real digit" and lets it pass, but the **condition is ignored**. Only by making the discriminator **also check "does the image match the label"** can we force the generator to **honestly obey the condition**.


<a id="3"></a>
## 3. 从零搭 cGAN：指定生成数字 ⭐ / Build cGAN: Generate Specified Digits

从零搭 cGAN：生成器和判别器都把 **one-hot 标签**拼接到输入。训练后, 我们**逐行指定 0~9**, 看每行是否生成对应的数字。
Build a cGAN from scratch: both G and D concatenate the **one-hot label**. After training, we **request 0–9 row by row** and check each row generates the right digit.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns, time
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
sns.set_theme(style="white"); torch.manual_seed(0)

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,),(0.5,))])
loader = DataLoader(Subset(datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tfm), range(20000)),
                    batch_size=128, shuffle=True, drop_last=True)
Z, NC = 100, 10
def onehot(y): return F.one_hot(y, NC).float()           # 标签转 one-hot 向量 / label → one-hot

# 生成器/判别器都把标签拼到输入 / both G and D concatenate the label
G = nn.Sequential(nn.Linear(Z+NC,256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
                  nn.Linear(256,512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2), nn.Linear(512,784), nn.Tanh())
D = nn.Sequential(nn.Linear(784+NC,512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
                  nn.Linear(512,256), nn.LeakyReLU(0.2), nn.Linear(256,1))
bce = nn.BCEWithLogitsLoss()
og = torch.optim.Adam(G.parameters(), 2e-4, betas=(0.5,0.999)); od = torch.optim.Adam(D.parameters(), 2e-4, betas=(0.5,0.999))
t0 = time.time()
for ep in range(70):
    for xb, yb in loader:
        b = xb.size(0); yoh = onehot(yb)
        z = torch.randn(b, Z); fake = G(torch.cat([z, yoh], 1))           # 生成器吃 噪声⊕标签 / noise⊕label
        # 训 D: 真图+真标签→真; 假图+标签→假 / train D with label concatenated
        od.zero_grad()
        loss_d = (bce(D(torch.cat([xb.view(b,-1), yoh], 1)), torch.ones(b,1)*0.9)
                  + bce(D(torch.cat([fake.detach(), yoh], 1)), torch.zeros(b,1)))
        loss_d.backward(); od.step()
        # 训 G: 让"假图+标签"被判为真 / train G to fool the conditioned D
        og.zero_grad(); loss_g = bce(D(torch.cat([fake, yoh], 1)), torch.ones(b,1)); loss_g.backward(); og.step()
print(f"cGAN 训练完成 ({time.time()-t0:.0f}s)")

# 指定生成: 每行是一个数字 0~9, 每列不同噪声 / generate: each row = a digit, columns = different noise
G.eval()
with torch.no_grad():
    rows = []
    for digit in range(10):
        y = torch.full((10,), digit); z = torch.randn(10, Z)
        rows.append(G(torch.cat([z, onehot(y)], 1)).view(-1,1,28,28))
    grid = torch.stack(rows)                              # (10 digits, 10 samples, 1,28,28)
fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for r in range(10):
    for c in range(10):
        axes[r,c].imshow(grid[r,c,0], cmap="gray"); axes[r,c].axis("off")
    axes[r,0].set_ylabel(str(r), fontsize=12, rotation=0, labelpad=10)
fig.suptitle("条件 GAN: 每行指定一个数字(0~9), 每列不同噪声 → 按指令可控生成", y=1.01)
plt.tight_layout(); plt.show()
print("每行确实生成了对应数字(同行不同写法=噪声z的作用) → 实现了可控生成")


<a id="4"></a>
## 4. 评估可控性 + 应用 + 小结 ⭐ / Evaluating Controllability & Applications

光看图主观。**定量评估可控性**：训练一个数字分类器, 让 cGAN 按指令生成各数字, 用分类器判断**生成的图是否真是被指定的那个数字**, 统计匹配率。
Eyeballing is subjective. **Quantify controllability:** train a digit classifier, have the cGAN generate each requested digit, and use the classifier to check whether the generated image **really is the requested digit**; report the match rate.


In [ ]:
# 训练一个 CNN 分类器(在真实MNIST上)用来检验 cGAN 生成的图是不是被指定的数字 / classifier to verify
clf = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(32*7*7,10))
oc = torch.optim.Adam(clf.parameters(), 1e-3)
clf_loader = DataLoader(Subset(datasets.MNIST(DATA_ROOT, train=True, transform=transforms.ToTensor()), range(20000)),
                        batch_size=128, shuffle=True)
for ep in range(4):
    for xb, yb in clf_loader: oc.zero_grad(); F.cross_entropy(clf(xb), yb).backward(); oc.step()

# 让 cGAN 生成各数字, 分类器判断是否匹配 / generate requested digits, classifier checks the match
G.eval(); clf.eval()
with torch.no_grad():
    req = torch.arange(10).repeat(30)                     # 每个数字请求30张 / 30 per digit
    z = torch.randn(len(req), Z)
    gen = G(torch.cat([z, onehot(req)], 1)).view(-1,1,28,28)
    gen01 = (gen + 1) / 2                                 # [-1,1] → [0,1] 配合分类器 / match classifier's range
    pred = clf(gen01).argmax(1)
    controllability = (pred == req).float().mean().item()
# 各数字的可控性 / per-digit controllability
per_digit = [(pred[req==d]==d).float().mean().item() for d in range(10)]
fig, ax = plt.subplots(figsize=(7,3.6))
ax.bar(range(10), per_digit, color="#39c"); ax.axhline(0.1, color="gray", ls="--", label="随机(0.1)")
ax.set_xlabel("被指定的数字"); ax.set_ylabel("分类器确认匹配的比例"); ax.set_xticks(range(10)); ax.legend()
ax.set_title(f"可控性: 生成的图被分类器确认为'指定数字'的比例 (总体 {controllability:.2f})")
plt.tight_layout(); plt.show()
print(f"总体可控性 = {controllability:.3f} (随机基线=0.10, 越高=越听指令)")
print("远高于随机 → cGAN 确实学会了按标签生成对应数字(判别器同时检查'图是否匹配标签'起了关键作用)")
print("诚实说明: 小MLP/CPU/有限训练下并非完美(部分数字更难/更易混); 真实cGAN(卷积+更久训练)可控性接近完美")


**条件生成的重要应用**(面试可举)：
**Important applications of conditional generation:**
- **pix2pix**:条件是**另一张图**——边缘图→照片、黑白→上色、地图→卫星图。判别器检查"输出图是否匹配输入图"。
  **pix2pix:** the condition is **another image** — edges→photo, grayscale→color, map→satellite. The discriminator checks "does the output match the input image."
- **CycleGAN**:无需配对数据的风格迁移(马↔斑马、夏↔冬)。
  **CycleGAN:** unpaired style transfer (horse↔zebra, summer↔winter).
- **文生图(text-to-image)**:条件是**文本**(经过文本编码器, 如 CLIP)——这正是现代文生图(及 Stable Diffusion 13.8)的核心思想: **用条件控制生成内容**。
  **Text-to-image:** the condition is **text** (via a text encoder like CLIP) — the core idea behind modern text-to-image (and Stable Diffusion, 13.8): **control generation with a condition.**

```
cGAN: 把条件y(标签/文本/图像)同时喂给生成器G(z,y)和判别器D(x,y) → 可控生成
判别器为何也要条件: 否则G可作弊(只生成最易骗过D的类别, 不管你要什么); D必须检查'图是否匹配条件'
加条件最简做法: 把条件向量(如one-hot)拼接到输入
评估可控性: 用分类器检验生成图是否真是被指定的类别; 远高于随机=可控成功
应用: pix2pix(图→图)/CycleGAN(无配对风格迁移)/文生图(文本条件, 通向Stable Diffusion)
本质: 从'随机生成'到'按条件生成'——生成模型走向实用的关键
```

### 💡 面试速查 / Interview cheat-sheet
1. **cGAN**: 条件y喂给G和D; G(z,y)按条件生成, z控制风格细节。
   cGAN: condition y to both G and D; G(z,y) generates per condition, z controls style.
2. **D 也要条件**: 否则G作弊忽略条件; D要查'图是否匹配条件'。
   D also conditioned: else G ignores the condition; D checks image-condition match.
3. **加条件**: 把条件向量拼接到输入(最简单有效)。
   Add condition: concatenate the condition vector to the input.
4. **评估**: 用分类器验证生成是否符合指定类别。
   Evaluate: a classifier verifies generations match the requested class.
5. **应用**: pix2pix/CycleGAN/文生图; 可控生成是实用关键。
   Apps: pix2pix/CycleGAN/text-to-image; controllability is key to usefulness.

### 下一节 / Next
**13.6 流模型(Normalizing Flows)**——又一条生成路线, 思路独特: 用**一系列可逆变换**把简单分布(高斯)精确地"掰"成复杂的数据分布。它的独门优势是**能精确计算似然**(VAE 只能近似, GAN 完全不能)。我们会在 2D 玩具数据上从零实现 RealNVP。
**13.6 Normalizing Flows** — another route with a unique idea: use **a series of invertible transformations** to exactly reshape a simple distribution (Gaussian) into the complex data distribution. Its unique advantage is **exact likelihood** (VAE only approximates, GAN can't at all). We'll implement RealNVP from scratch on 2D toy data.
